# 06b - Export Submission

CPU only, final stage. Export the frozen winner as agent.py and submission.zip, then run the final submission checks. Nothing is uploaded automatically.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Load the Frozen Winner

Require successful completion of 06a and verify its frozen sources, configuration, checkpoint and evaluation evidence. The API remains get_move(fen: str, time_left_ms: int) -> str.

In [ ]:
import torch
torch.set_num_threads(1)
from chess_rl.export import freeze_requirement
selection, checkpoint = freeze_requirement(PROJECT_ROOT, cfg["run_id"])
print("Winner:", selection["selected_id"], "Checkpoint:", checkpoint)

## Optional ONNX Dependencies

Only needed for an explicitly configured neural ONNX export.

In [ ]:
if checkpoint is not None and selection["config"]["export"]["format"] == "onnx":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "onnx", "onnxruntime"])

## Export and Final Checks

The selected neural or classical export runs import/API checks, legal moves, low-clock timing, CPU-only inference, package integrity/size and agent.py-at-ZIP-root checks. It also plays final local games. A failed report requires investigation; this stage does not retrain.

In [ ]:
from chess_rl.export import final_selected_checks
report = final_selected_checks(PROJECT_ROOT, cfg["run_id"], live_games=16)
print("Status:", report["status"])
print("Submission:", report.get("archive", report.get("submission")))
print("Checks:", report["checks"])
print("Report:", PROJECT_ROOT / "results" / cfg["run_id"] / "final_compliance.json")

## Submission

Use the submission.zip reported above for manual platform submission. Local timing is not a measurement of the platform host, and local success is not platform acceptance.